In [1]:
from databricks import sql
import polars as pl
import pyodbc

# Synapse Connection
synapse_server = 'syn-rt-prd-unity.sql.azuresynapse.net'
synapse_database = 'syndwrtprdunity'
synapse_username = 'byambadorjme@riotinto.com'
synapse_driver = '{ODBC Driver 17 for SQL Server}'
synapse_schema_name = 'HSTG_V'
synapse_table_name = 'VW_HSTG_UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENT'

synapse_connection_string = f"""
    DRIVER={synapse_driver};
    SERVER={synapse_server};
    DATABASE={synapse_database};
    UID={synapse_username};
    Authentication=ActiveDirectoryInteractive;
    Encrypt=yes;
    TrustServerCertificate=no;
    Connection Timeout=30;
"""

# Databricks Connection
databricks_access_token = os.environ["DATABRICKS_TOKEN"]
databricks_server_hostname = 'adb-2439498039309203.3.azuredatabricks.net'
databricks_http_path = '/sql/1.0/warehouses/e7d134c4348ac8b5'
databricks_catalog_name = '_ogr_azr_syd'
databricks_schema_name = 'gensd03_sicom'
databricks_table_name = 'dbo_hsp_suivietatequipement'

In [2]:
# Show up to 100 rows and 50 columns
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(50)

# Show full content of long strings (e.g., 100 characters)
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

In [3]:
databricks_to_synapse_type_map = {
    "TINYINT": "TINYINT",
    "SMALLINT": "SMALLINT",
    "INT": "INT",
    "INTEGER": "INT",
    "BIGINT": "BIGINT",
    "FLOAT": "REAL",
    "REAL": "REAL",
    "DOUBLE": "FLOAT",
    "DECIMAL": "DECIMAL",
    "NUMERIC": "NUMERIC",
    "STRING": "VARCHAR",
    "VARCHAR": "VARCHAR",
    "CHAR": "CHAR",
    "BINARY": "VARBINARY(8000)",
    "DATE": "DATE",
    "TIMESTAMP": "DATETIME2",
    "TIMESTAMP_NTZ": "DATETIME2",
    "TIMESTAMP_LTZ": "DATETIMEOFFSET",
    "BOOLEAN": "BIT"
}

databricks_to_polars_type_map = {
    "TINYINT": pl.Int8,
    "SMALLINT": pl.Int16,
    "INT": pl.Int32,
    "INTEGER": pl.Int32,
    "BIGINT": pl.Int64,
    "FLOAT": pl.Float32,
    "REAL": pl.Float32,
    "DOUBLE": pl.Float64,
    "DECIMAL": pl.Decimal(38, 18),   # adjust precision/scale if needed
    "NUMERIC": pl.Decimal(38, 18),   # adjust precision/scale if needed
    "STRING": pl.String,
    "VARCHAR": pl.String,
    "CHAR": pl.String,
    "BINARY": pl.Binary,
    "DATE": pl.Date,
    "TIMESTAMP": pl.Datetime("us"),
    "TIMESTAMP_NTZ": pl.Datetime("us"),
    "TIMESTAMP_LTZ": pl.Datetime("us", "UTC"),
    "BOOLEAN": pl.Boolean
}

In [4]:
type_map_df = pl.DataFrame(
    [
        {"databricks_data_type": k, "synapse_expected_type": v}
        for k, v in databricks_to_synapse_type_map.items()
    ]
).sort("databricks_data_type")

In [5]:
synapse_conn = pyodbc.connect(synapse_connection_string)

In [ ]:
synapse_schema_query = f"""
select 
    column_name as synapse_column_name,
    upper(data_type) as synapse_data_type
from information_schema.columns
where table_schema = '{synapse_schema_name}'
and table_name = '{synapse_table_name}'
and column_name not like 'OMD%'
and column_name not like 'HSTG%'
order by ordinal_position
"""

synapse_schema_df = pl.read_database(synapse_schema_query, synapse_conn)

In [ ]:
# Get Databricks Column Names and Data Types
print("Getting Databricks schema...")
databricks_conn = sql.connect(
    server_hostname=databricks_server_hostname,
    http_path=databricks_http_path,
    access_token=databricks_access_token
)

databricks_schema_query = f"""
            select 
                upper(column_name) as databricks_column_name, 
                upper(data_type) as databricks_data_type
            from {databricks_catalog_name}.information_schema.columns 
            where table_name = '{databricks_table_name}' 
            and table_schema = '{databricks_schema_name}'
            and column_name not like 'rtlh%'
"""
databricks_schema_df = pl.read_database(databricks_schema_query, databricks_conn)

In [ ]:
schema_comparison_df = synapse_schema_df.join(
    databricks_schema_df,
    left_on="synapse_column_name",
    right_on="databricks_column_name",
    how="full"
).select([
    pl.col("synapse_column_name"),
    pl.col("synapse_data_type"),
    pl.col("databricks_column_name"),
    pl.col("databricks_data_type")
]).join(
    type_map_df,
    left_on="databricks_data_type",
    right_on="databricks_data_type",
    how="left"
)

In [ ]:
synapse_cols = ", ".join(
    f"{c} as SYNAPSE_{c}"
    for c in schema_comparison_df["synapse_column_name"].to_list()
)
synapse_full_table_query = f"""
    select {synapse_cols}
    from {synapse_schema_name}.{synapse_table_name}
    where omd_current_record_indicator = 'Y'
    and omd_deleted_record_indicator = 'N'
"""

synapse_full_table_df = pl.read_database(synapse_full_table_query, synapse_conn)

In [ ]:
databricks_cols = ", ".join(
    f"{c} as DATABRICKS_{c}"
    for c in schema_comparison_df["databricks_column_name"].to_list()
)
databricks_full_table_query = f"""
    select {databricks_cols}
    from {databricks_catalog_name}.{databricks_schema_name}.{databricks_table_name}
"""

databricks_full_table_df = pl.read_database(databricks_full_table_query, databricks_conn)

In [ ]:
tz_cols = [
    name for name, dtype in databricks_full_table_df.schema.items()
    if dtype == pl.Datetime and getattr(dtype, "time_zone", None) is not None
]

databricks_full_table_df = databricks_full_table_df.with_columns([pl.col(c).dt.replace_time_zone(None) for c in tz_cols])

In [ ]:
import polars as pl

INT_LIKE = {
    pl.Boolean, pl.Int8, pl.Int16, pl.Int32, pl.Int64,
    pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64
}
FLOAT_LIKE = {pl.Float32, pl.Float64}

# 1) Align to the same logical column names
pairs = (
    schema_comparison_df
    .select(["synapse_column_name", "databricks_column_name"])
    .drop_nulls()
    .to_dicts()
)

syn_aligned = synapse_full_table_df.select([
    pl.col(f"SYNAPSE_{r['synapse_column_name']}").alias(r["synapse_column_name"])
    for r in pairs
])

dbx_aligned = databricks_full_table_df.select([
    pl.col(f"DATABRICKS_{r['databricks_column_name']}").alias(r["synapse_column_name"])
    for r in pairs
])

# 2) Normalize dtypes (ints -> Int64, floats -> Float64, datetimes -> naive us)
def normalize_types(df: pl.DataFrame) -> pl.DataFrame:
    exprs = []
    for c, d in df.schema.items():
        if d in INT_LIKE:
            exprs.append(pl.col(c).cast(pl.Int64).alias(c))
        elif d in FLOAT_LIKE:
            exprs.append(pl.col(c).cast(pl.Float64).alias(c))
        elif str(d).startswith("Datetime"):
            e = pl.col(c)
            if "time_zone" in str(d):
                e = e.dt.replace_time_zone(None)
            exprs.append(e.dt.cast_time_unit("us").alias(c))
    return df.with_columns(exprs)

syn_norm = normalize_types(syn_aligned)
dbx_norm = normalize_types(dbx_aligned)

# 3) Hash after normalization
syn_hashed = syn_norm.with_columns(pl.struct(pl.all()).hash().alias("row_hash"))
dbx_hashed = dbx_norm.with_columns(pl.struct(pl.all()).hash().alias("row_hash"))

print("Match:", syn_hashed["row_hash"].sort().equals(dbx_hashed["row_hash"].sort()))

# counts
syn_counts = (
    syn_hashed
    .group_by("row_hash")
    .len()
    .rename({"len": "syn_cnt"})
)

dbx_counts = (
    dbx_hashed
    .group_by("row_hash")
    .len()
    .rename({"len": "dbx_cnt"})
)

cmp = (
    syn_counts.join(dbx_counts, on="row_hash", how="full")
    .with_columns(
        pl.col("syn_cnt").fill_null(0).cast(pl.Int64),
        pl.col("dbx_cnt").fill_null(0).cast(pl.Int64),
    )
    .with_columns(
        pl.min_horizontal("syn_cnt", "dbx_cnt").alias("matched_cnt")
    )
)

# Use sums from counts (guaranteed same universe as matched_cnt)
syn_total = int(syn_counts["syn_cnt"].sum())
dbx_total = int(dbx_counts["dbx_cnt"].sum())
matched_rows = int(cmp["matched_cnt"].sum())

# Defensive clamp (prevents >100% if anything odd leaks in)
matched_rows = min(matched_rows, syn_total, dbx_total)

pct_of_synapse = 100.0 * matched_rows / syn_total if syn_total else 100.0
pct_of_databricks = 100.0 * matched_rows / dbx_total if dbx_total else 100.0
pct_overall = 100.0 * matched_rows / max(syn_total, dbx_total) if max(syn_total, dbx_total) else 100.0

print(f"Matched rows: {matched_rows}")
print(f"Synapse rows: {syn_total}")
print(f"Databricks rows: {dbx_total}")
print(f"Match % of Synapse: {pct_of_synapse:.10f}%")
print(f"Match % of Databricks: {pct_of_databricks:.10f}%")
print(f"Overall Match %: {pct_overall:.10f}%")

In [ ]:
from databricks import sql
import polars as pl

def compare_synapse_vs_databricks(
    databricks_conn,
    synapse_conn,
    synapse_schema_name,
    synapse_table_name,
    databricks_catalog_name,
    databricks_schema_name,
    databricks_table_name,
):
    # Type normalization helpers
    def base_type(dtype: str) -> str:
        return str(dtype).split("(")[0].strip().upper() if dtype is not None else None

    databricks_to_synapse_type_map = {
        "TINYINT": "TINYINT",
        "SMALLINT": "SMALLINT",
        "INT": "INT",
        "INTEGER": "INT",
        "BIGINT": "BIGINT",
        "FLOAT": "REAL",
        "REAL": "REAL",
        "DOUBLE": "FLOAT",
        "DECIMAL": "DECIMAL",
        "NUMERIC": "NUMERIC",
        "STRING": "VARCHAR",
        "VARCHAR": "VARCHAR",
        "CHAR": "CHAR",
        "BINARY": "VARBINARY",
        "DATE": "DATE",
        "TIMESTAMP": "DATETIME2",
        "TIMESTAMP_NTZ": "DATETIME2",
        "TIMESTAMP_LTZ": "DATETIMEOFFSET",
        "BOOLEAN": "BIT",
    }

    # 1) Read schema metadata
    synapse_schema_query = f"""
        select
            upper(column_name) as column_name,
            upper(data_type) as synapse_data_type
        from information_schema.columns
        where table_schema = '{synapse_schema_name}'
          and table_name = '{synapse_table_name}'
          and column_name not like 'OMD%'
          and column_name not like 'HSTG%'
        order by ordinal_position
    """

    databricks_schema_query = f"""
        select
            upper(column_name) as column_name,
            upper(data_type) as databricks_data_type
        from {databricks_catalog_name}.information_schema.columns
        where table_schema = '{databricks_schema_name}'
          and table_name = '{databricks_table_name}'
          and column_name not like 'rtlh%'
    """

    synapse_schema_df = pl.read_database(synapse_schema_query, synapse_conn)
    databricks_schema_df = pl.read_database(databricks_schema_query, databricks_conn)

    # Map Databricks type -> expected Synapse type
    databricks_schema_df = databricks_schema_df.with_columns(
        pl.col("databricks_data_type")
        .map_elements(lambda x: databricks_to_synapse_type_map.get(base_type(x), None), return_dtype=pl.Utf8)
        .alias("synapse_expected_type")
    )

    # 2) Compare schema
    schema_comparison_df = synapse_schema_df.join(
        databricks_schema_df,
        on="column_name",
        how="full"
    )

    schema_mismatches_df = schema_comparison_df.filter(
        pl.col("synapse_data_type").is_null()
        | pl.col("databricks_data_type").is_null()
        | (
            pl.col("synapse_data_type").map_elements(base_type, return_dtype=pl.Utf8)
            != pl.col("synapse_expected_type").map_elements(base_type, return_dtype=pl.Utf8)
        )
    )

    schema_matches = schema_mismatches_df.height == 0

    # 3) Row count check (same notebook logic)
    synapse_count_query = f"""
        select count(*) as row_count
        from {synapse_schema_name}.{synapse_table_name}
        where omd_current_record_indicator = 'Y'
          and omd_deleted_record_indicator = 'N'
    """

    databricks_count_query = f"""
        select count(*) as row_count
        from {databricks_catalog_name}.{databricks_schema_name}.{databricks_table_name}
    """

    synapse_row_count = int(pl.read_database(synapse_count_query, synapse_conn)["row_count"][0])
    databricks_row_count = int(pl.read_database(databricks_count_query, databricks_conn)["row_count"][0])
    row_count_matches = synapse_row_count == databricks_row_count

    # 4) Full match on common columns with detailed alignment and normalization
    common_cols = sorted(
        set(synapse_schema_df["column_name"].to_list())
        & set(databricks_schema_df["column_name"].to_list())
    )

    if not common_cols:
        return {
            "schema_matches": schema_matches,
            "row_count_matches": row_count_matches,
            "full_match": False,
            "reason": "No common columns found for full comparison.",
            "synapse_row_count": synapse_row_count,
            "databricks_row_count": databricks_row_count,
            "schema_mismatches_df": schema_mismatches_df,
            "schema_comparison_df": schema_comparison_df,
            "matched_rows": 0,
            "pct_of_synapse": 0.0,
            "pct_of_databricks": 0.0,
            "pct_overall": 0.0,
        }

    synapse_cols_sql = ", ".join([f"[{c}] as [SYNAPSE_{c}]" for c in common_cols])
    databricks_cols_sql = ", ".join([f"`{c}` as `DATABRICKS_{c}`" for c in common_cols])

    synapse_full_query = f"""
        select {synapse_cols_sql}
        from {synapse_schema_name}.{synapse_table_name}
        where omd_current_record_indicator = 'Y'
          and omd_deleted_record_indicator = 'N'
    """

    databricks_full_query = f"""
        select {databricks_cols_sql}
        from {databricks_catalog_name}.{databricks_schema_name}.{databricks_table_name}
    """

    synapse_full_table_df = pl.read_database(synapse_full_query, synapse_conn)
    databricks_full_table_df = pl.read_database(databricks_full_query, databricks_conn)

    # Normalize timezone-aware datetime columns in Databricks data
    tz_cols = [
        name for name, dtype in databricks_full_table_df.schema.items()
        if dtype == pl.Datetime and getattr(dtype, "time_zone", None) is not None
    ]
    if tz_cols:
        databricks_full_table_df = databricks_full_table_df.with_columns(
            [pl.col(c).dt.replace_time_zone(None) for c in tz_cols]
        )

    # ============ Enhanced Matching Logic ============
    INT_LIKE = {
        pl.Boolean, pl.Int8, pl.Int16, pl.Int32, pl.Int64,
        pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64
    }
    FLOAT_LIKE = {pl.Float32, pl.Float64}

    # 1) Align to the same logical column names
    pairs = [
        {"synapse_column_name": c, "databricks_column_name": c}
        for c in common_cols
    ]

    syn_aligned = synapse_full_table_df.select([
        pl.col(f"SYNAPSE_{r['synapse_column_name']}").alias(r["synapse_column_name"])
        for r in pairs
    ])

    dbx_aligned = databricks_full_table_df.select([
        pl.col(f"DATABRICKS_{r['databricks_column_name']}").alias(r["synapse_column_name"])
        for r in pairs
    ])

    # 2) Normalize dtypes (ints -> Int64, floats -> Float64, datetimes -> naive us)
    def normalize_types(df: pl.DataFrame) -> pl.DataFrame:
        exprs = []
        for c, d in df.schema.items():
            if d in INT_LIKE:
                exprs.append(pl.col(c).cast(pl.Int64).alias(c))
            elif d in FLOAT_LIKE:
                exprs.append(pl.col(c).cast(pl.Float64).alias(c))
            elif str(d).startswith("Datetime"):
                e = pl.col(c)
                if "time_zone" in str(d):
                    e = e.dt.replace_time_zone(None)
                exprs.append(e.dt.cast_time_unit("us").alias(c))
        return df.with_columns(exprs) if exprs else df

    syn_norm = normalize_types(syn_aligned)
    dbx_norm = normalize_types(dbx_aligned)

    # 3) Hash after normalization
    syn_hashed = syn_norm.with_columns(pl.struct(pl.all()).hash().alias("row_hash"))
    dbx_hashed = dbx_norm.with_columns(pl.struct(pl.all()).hash().alias("row_hash"))

    full_match_exact = syn_hashed["row_hash"].sort().equals(dbx_hashed["row_hash"].sort())

    # Counts and detailed matching
    syn_counts = (
        syn_hashed
        .group_by("row_hash")
        .len()
        .rename({"len": "syn_cnt"})
    )

    dbx_counts = (
        dbx_hashed
        .group_by("row_hash")
        .len()
        .rename({"len": "dbx_cnt"})
    )

    cmp = (
        syn_counts.join(dbx_counts, on="row_hash", how="full")
        .with_columns(
            pl.col("syn_cnt").fill_null(0).cast(pl.Int64),
            pl.col("dbx_cnt").fill_null(0).cast(pl.Int64),
        )
        .with_columns(
            pl.min_horizontal("syn_cnt", "dbx_cnt").alias("matched_cnt")
        )
    )

    # Use sums from counts (guaranteed same universe as matched_cnt)
    syn_total = int(syn_counts["syn_cnt"].sum())
    dbx_total = int(dbx_counts["dbx_cnt"].sum())
    matched_rows = int(cmp["matched_cnt"].sum())

    # Defensive clamp (prevents >100% if anything odd leaks in)
    matched_rows = min(matched_rows, syn_total, dbx_total)

    pct_of_synapse = 100.0 * matched_rows / syn_total if syn_total else 100.0
    pct_of_databricks = 100.0 * matched_rows / dbx_total if dbx_total else 100.0
    pct_overall = 100.0 * matched_rows / max(syn_total, dbx_total) if max(syn_total, dbx_total) else 100.0

    return {
        "schema_matches": schema_matches,
        "row_count_matches": row_count_matches,
        "full_match": full_match_exact,
        "synapse_row_count": synapse_row_count,
        "databricks_row_count": databricks_row_count,
        "schema_mismatches_df": schema_mismatches_df,
        "schema_comparison_df": schema_comparison_df,
        "common_columns_compared": common_cols,
        "matched_rows": matched_rows,
        "syn_total": syn_total,
        "dbx_total": dbx_total,
        "pct_of_synapse": pct_of_synapse,
        "pct_of_databricks": pct_of_databricks,
        "pct_overall": pct_overall,
    }

| # | Synapse Schema | Synapse Table | Databricks Table |
|---|---|---|---|
| 1 | DAL_V | VW_UDLGENSD03_DBO_HSP_EQUIPEMENT_DEF | dbo_hsp_equipement_def |
| 2 | DAL_V | VW_UDLGENSD03_DBO_HSP_MODELEEQUIPEMENT_DEF | dbo_hsp_modeleequipement_def |
| 3 | DAL_V | VW_UDLGENSD03_DBO_HSP_GROUPEEQUIPEMENT_DEF | dbo_hsp_groupequipement_def |
| 4 | DAL_V | VW_UDLGENSD03_DBO_HSP_FAMILLEEQUIPEMENT_DEF | dbo_hsp_familleequipement_def |
| 5 | DAL_V | VW_UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENT | dbo_hsp_suivietatequipement |
| 6 | HSTG_V | VW_HSTG_UDLGENSD03_DBO_GEN_CODEIPT_DEF | dbo_gen_codeipt_def |
| 7 | HSTG_V | VW_HSTG_UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENTROWNUMBER_VIEW | dbo_hsp_suivietatequipementrownumber_view |

In [ ]:
# UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENTROWNUMBER_VIEW
result = compare_synapse_vs_databricks(
    databricks_conn=databricks_conn,
    synapse_conn=synapse_conn,
    synapse_schema_name="HSTG_V",
    synapse_table_name="VW_HSTG_UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENTROWNUMBER_VIEW",
    databricks_catalog_name="_ogr_azr_syd",
    databricks_schema_name="gensd03_sicom",
    databricks_table_name="dbo_hsp_suivietatequipementrownumber_view",
)

print(f"Full match: {result['full_match']}")
print(f"Synapse row count: {result['synapse_row_count']}")
print(f"Databricks row count: {result['databricks_row_count']}")
print(f"Matched: {result['matched_rows']}/{result['syn_total']} ({result['pct_of_synapse']:.10f}%)")
result["schema_mismatches_df"]

In [ ]:
# UDLGENSD03_DBO_GEN_CODEIPT_DEF
result = compare_synapse_vs_databricks(
    databricks_conn=databricks_conn,
    synapse_conn=synapse_conn,
    synapse_schema_name="HSTG_V",
    synapse_table_name="VW_HSTG_UDLGENSD03_DBO_GEN_CODEIPT_DEF",
    databricks_catalog_name="_ogr_azr_syd",
    databricks_schema_name="gensd03_sicom",
    databricks_table_name="dbo_gen_codeipt_def",
)

print(f"Full match: {result['full_match']}")
print(f"Synapse row count: {result['synapse_row_count']}")
print(f"Databricks row count: {result['databricks_row_count']}")
print(f"Matched: {result['matched_rows']}/{result['syn_total']} ({result['pct_of_synapse']:.10f}%)")
result["schema_mismatches_df"]

In [ ]:
# UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENT
result = compare_synapse_vs_databricks(
    databricks_conn=databricks_conn,
    synapse_conn=synapse_conn,
    synapse_schema_name="HSTG_V",
    synapse_table_name="VW_HSTG_UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENT",
    databricks_catalog_name="_ogr_azr_syd",
    databricks_schema_name="gensd03_sicom",
    databricks_table_name="dbo_hsp_suivietatequipement",
)

print(f"Full match: {result['full_match']}")
print(f"Synapse row count: {result['synapse_row_count']}")
print(f"Databricks row count: {result['databricks_row_count']}")
print(f"Matched: {result['matched_rows']}/{result['syn_total']} ({result['pct_of_synapse']:.10f}%)")
result["schema_mismatches_df"]

In [ ]:
# UDLGENSD03_DBO_HSP_GROUPEEQUIPEMENT_DEF
result = compare_synapse_vs_databricks(
    databricks_conn=databricks_conn,
    synapse_conn=synapse_conn,
    synapse_schema_name="HSTG_V",
    synapse_table_name="VW_HSTG_UDLGENSD03_DBO_HSP_GROUPEEQUIPEMENT_DEF",
    databricks_catalog_name="_ogr_azr_syd",
    databricks_schema_name="gensd03_sicom",
    databricks_table_name="dbo_hsp_groupeequipement_def",
)

print(f"Full match: {result['full_match']}")
print(f"Matched: {result['matched_rows']}/{result['syn_total']} ({result['pct_of_synapse']:.2f}%)")
result["schema_mismatches_df"]

In [ ]:
# UDLGENSD03_DBO_HSP_MODELEEQUIPEMENT_DEF
result = compare_synapse_vs_databricks(
    databricks_conn=databricks_conn,
    synapse_conn=synapse_conn,
    synapse_schema_name="HSTG_V",
    synapse_table_name="VW_HSTG_UDLGENSD03_DBO_HSP_MODELEEQUIPEMENT_DEF",
    databricks_catalog_name="_ogr_azr_syd",
    databricks_schema_name="gensd03_sicom",
    databricks_table_name="dbo_hsp_modeleequipement_def",
)

print(f"Full match: {result['full_match']}")
print(f"Matched: {result['matched_rows']}/{result['syn_total']} ({result['pct_of_synapse']:.2f}%)")
result["schema_mismatches_df"]

In [ ]:
# VW_UDLGENSD03_DBO_HSP_EQUIPEMENT_DEF
result = compare_synapse_vs_databricks(
    databricks_conn=databricks_conn,
    synapse_conn=synapse_conn,
    synapse_schema_name="HSTG_V",
    synapse_table_name="VW_HSTG_UDLGENSD03_DBO_HSP_EQUIPEMENT_DEF",
    databricks_catalog_name="_ogr_azr_syd",
    databricks_schema_name="gensd03_sicom",
    databricks_table_name="dbo_hsp_equipement_def",
)

print(f"Full match: {result['full_match']}")
print(f"Matched: {result['matched_rows']}/{result['syn_total']} ({result['pct_of_synapse']:.2f}%)")
result["schema_mismatches_df"]